# 업계동향 기사 7개 편집 의도 분류

기존 `news_industry_trends_embeddings.npy`를 재사용해 과거 업계동향 기사를 7개 편집 의도로 묶습니다. 모델 재학습이나 기사 재임베딩은 수행하지 않습니다.

산출물:
- `model/output/news_industry_trends_intent_labels.csv`: 검토·수정용 라벨
- `model/embeddings/industry_intents/*.npy`: 의도별 대표 기사 벡터
- `industry_intents_metadata.json`: 벡터와 기사 연결 정보
- `manifest.json`: 생성 조건과 묶음별 통계

In [ ]:
from __future__ import annotations

from collections import Counter
from datetime import datetime, timezone
import csv
import json
from pathlib import Path
import re

import numpy as np

MODEL_DIR = Path.cwd().resolve()
if MODEL_DIR.name != 'model':
    candidate = MODEL_DIR / 'model'
    if candidate.exists():
        MODEL_DIR = candidate

EMBEDDING_DIR = MODEL_DIR / 'embeddings'
OUTPUT_DIR = MODEL_DIR / 'output'
INTENT_DIR = EMBEDDING_DIR / 'industry_intents'
SOURCE_VECTOR_PATH = EMBEDDING_DIR / 'news_industry_trends_embeddings.npy'
SOURCE_METADATA_PATH = EMBEDDING_DIR / 'news_industry_trends_metadata.json'
SOURCE_CSV_PATH = OUTPUT_DIR / 'news_industry_trends_enriched.csv'
LABEL_CSV_PATH = OUTPUT_DIR / 'news_industry_trends_intent_labels.csv'
METADATA_PATH = INTENT_DIR / 'industry_intents_metadata.json'
MANIFEST_PATH = INTENT_DIR / 'manifest.json'

CLASSIFICATION_VERSION = 1
SEED_LIMIT = 250
REFERENCE_LIMIT = 100
SECONDARY_MARGIN = 0.04
SECONDARY_MIN_SCORE = 0.50
MMR_RELEVANCE = 0.75
DUPLICATE_COSINE = 0.94

INTENT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('모델 폴더:', MODEL_DIR)

## 1. 편집 의도와 고신뢰 시드 규칙

키워드는 최종 추천 포함 조건이 아니라 초기 의미 중심을 만드는 용도입니다. 제목 일치는 요약 일치보다 두 배 강하게 반영합니다.

In [ ]:
INTENTS = {
    'model_platform': {
        'label': '모델·플랫폼·빅테크',
        'description': '생성형 AI, 기반 모델, AI 플랫폼과 글로벌 빅테크 모델 경쟁',
        'keywords': {
            '생성형 ai': 2.5, '거대언어모델': 3.0, 'llm': 3.0,
            '기반 모델': 3.0, '파운데이션 모델': 3.0, 'ai 모델': 2.5,
            'ai 플랫폼': 2.5, '멀티모달': 2.5, '오픈ai': 2.0,
            'openai': 2.0, '챗gpt': 2.0, 'chatgpt': 2.0,
            '제미나이': 2.0, 'gemini': 2.0, '클로드': 2.0,
            '앤트로픽': 2.0, 'anthropic': 2.0, '라마': 2.0,
        },
    },
    'semiconductor_infra': {
        'label': '반도체·인프라',
        'description': 'AI 반도체, GPU, 데이터센터, 클라우드와 연산 인프라',
        'keywords': {
            'ai 반도체': 3.0, 'ai 칩': 3.0, 'ai 가속기': 3.0,
            'gpu': 2.5, 'hbm': 3.0, 'npu': 3.0, '데이터센터': 2.5,
            '데이터 센터': 2.5, 'ai 인프라': 2.5, 'ai 클라우드': 2.5,
            '클라우드 인프라': 2.0, '온디바이스 ai': 2.5,
            '엣지 ai': 2.5, '엣지 컴퓨팅': 2.5, '엔비디아': 1.5,
        },
    },
    'enterprise_adoption': {
        'label': '기업 도입·업무 혁신',
        'description': '기업용 AI, AI 에이전트, AX와 업무 프로세스 혁신',
        'keywords': {
            '기업용 ai': 3.0, '엔터프라이즈 ai': 3.0, 'ai 에이전트': 2.5,
            '에이전틱 ai': 2.5, 'ai 비서': 2.0, '업무 자동화': 2.5,
            '업무 혁신': 2.5, 'ax': 2.0, '코파일럿': 2.0,
            'copilot': 2.0, '기업 ai': 2.0, '산업 ai': 1.5,
            'ai 전환': 2.0, 'ai 솔루션': 1.5,
        },
    },
    'physical_manufacturing': {
        'label': '제조·로봇·피지컬 AI',
        'description': '제조 AI, 로봇, 피지컬 AI, 스마트팩토리와 자율주행',
        'keywords': {
            '피지컬 ai': 3.0, '제조 ai': 3.0, '산업용 ai': 2.0,
            '로봇': 2.0, '휴머노이드': 2.5, '스마트팩토리': 2.5,
            '스마트 팩토리': 2.5, '자율주행': 2.5, 'sdv': 2.5,
            '디지털트윈': 2.5, '디지털 트윈': 2.5,
            '컴퓨터비전': 2.0, '컴퓨터 비전': 2.0,
            '공정 자동화': 2.5, '공장 자동화': 2.5,
        },
    },
    'vertical_applications': {
        'label': '산업별 응용',
        'description': '의료, 바이오, 국방, 에너지, 커머스와 콘텐츠 분야의 AI 응용',
        'keywords': {
            '의료 ai': 3.0, '헬스케어 ai': 3.0, '바이오 ai': 3.0,
            '신약개발 ai': 3.0, '제약 ai': 3.0, '국방 ai': 3.0,
            '에너지 ai': 3.0, '금융 ai': 2.5, '커머스 ai': 2.5,
            '콘텐츠 ai': 2.5, '교육 ai': 2.5, '법률 ai': 2.5,
            '리걸 ai': 2.5, '농업 ai': 2.5, '기후 ai': 2.5,
            'ai 진단': 2.0, 'ai 의료': 2.5,
        },
    },
    'market_supply_chain': {
        'label': '시장·경쟁·공급망',
        'description': 'AI 시장 경쟁, 인수합병, 대규모 투자, 공급망과 수출통제',
        'keywords': {
            'ai 시장': 2.5, '시장 점유율': 2.5, '시장 전망': 2.0,
            '인수합병': 3.0, 'm&a': 3.0, '인수': 1.5, '합병': 2.0,
            '공급망': 3.0, '수출통제': 3.0, '수출 통제': 3.0,
            '빅테크 경쟁': 2.5, 'ai 경쟁': 2.0, '패권 경쟁': 2.5,
            '대규모 투자': 2.5, '조원 투자': 2.5, '설비투자': 2.5,
            'ai 매출': 2.0, 'ai 수익': 2.0,
        },
    },
    'regulation_social': {
        'label': '규제·안전·사회 영향',
        'description': 'AI 규제, 개인정보, 저작권, 안전, 윤리와 고용·사회 영향',
        'keywords': {
            'ai 규제': 3.0, '인공지능 규제': 3.0, 'ai 기본법': 3.0,
            '개인정보': 2.5, '프라이버시': 2.5, 'ai 저작권': 3.0,
            '저작권': 2.0, 'ai 안전': 3.0, 'ai 윤리': 3.0,
            '딥페이크': 2.5, '일자리': 2.0, '고용': 1.5,
            '해고': 2.0, '노동시장': 2.5, 'ai 소송': 2.5,
            '알고리즘 책임': 2.5, 'ai 위험': 2.5,
        },
    },
}

INTENT_SLUGS = list(INTENTS)
print('편집 의도:', [INTENTS[slug]['label'] for slug in INTENT_SLUGS])

## 2. 기존 벡터와 기사 메타데이터 로드

In [ ]:
vectors = np.load(SOURCE_VECTOR_PATH, mmap_mode='r')
metadata_document = json.loads(SOURCE_METADATA_PATH.read_text(encoding='utf-8'))
rows = metadata_document['rows']

if vectors.ndim != 2 or len(vectors) != len(rows):
    raise ValueError(f'벡터/메타데이터 행 불일치: {vectors.shape} vs {len(rows)}')

enriched_by_id = {}
with SOURCE_CSV_PATH.open('r', encoding='utf-8-sig', newline='') as handle:
    for item in csv.DictReader(handle):
        enriched_by_id[item.get('article_id', '')] = item

articles = []
for index, row in enumerate(rows):
    enriched = enriched_by_id.get(row.get('article_id', ''), {})
    articles.append({
        **row,
        'vector_index': index,
        'digest_date': enriched.get('digest_date', ''),
        'summary': enriched.get('summary', ''),
        'publisher': enriched.get('publisher', ''),
        'topic': enriched.get('topic', ''),
    })

normalized_vectors = np.asarray(vectors, dtype=np.float32)
norms = np.linalg.norm(normalized_vectors, axis=1, keepdims=True)
normalized_vectors = normalized_vectors / np.clip(norms, 1e-12, None)
print('기사/벡터:', len(articles), normalized_vectors.shape)
print('요약 누락:', sum(not article['summary'] for article in articles))

## 3. 규칙 시드 생성과 의미 중심 계산

각 묶음에서 규칙 점수가 높은 기사를 시드로 고른 뒤 기존 결합 벡터의 정규화 평균을 의미 중심으로 사용합니다.

In [ ]:
WORD_CHAR = re.compile(r'[0-9A-Za-z가-힣]')

def keyword_present(text: str, keyword: str) -> bool:
    text = text.casefold()
    keyword = keyword.casefold().strip()
    if not keyword:
        return False
    start = 0
    while True:
        index = text.find(keyword, start)
        if index < 0:
            return False
        left_ok = index == 0 or not WORD_CHAR.match(text[index - 1])
        end = index + len(keyword)
        right_ok = end == len(text) or not WORD_CHAR.match(text[end])
        if left_ok and right_ok:
            return True
        start = index + 1

def rule_score(article: dict, intent: dict) -> float:
    title = article.get('title', '')
    summary = article.get('summary', '')
    score = 0.0
    for keyword, weight in intent['keywords'].items():
        if keyword_present(title, keyword):
            score += 2.0 * weight
        elif keyword_present(summary, keyword):
            score += weight
    return score

raw_rule_scores = np.zeros((len(articles), len(INTENT_SLUGS)), dtype=np.float32)
for row_index, article in enumerate(articles):
    for intent_index, slug in enumerate(INTENT_SLUGS):
        raw_rule_scores[row_index, intent_index] = rule_score(article, INTENTS[slug])

centroids = []
seed_indices_by_intent = {}
for intent_index, slug in enumerate(INTENT_SLUGS):
    scores = raw_rule_scores[:, intent_index]
    positive = np.flatnonzero(scores > 0)
    if len(positive) < 10:
        raise ValueError(f'{slug}: 의미 중심을 만들 시드가 부족합니다 ({len(positive)}건)')
    ordered = positive[np.argsort(scores[positive])[::-1]]
    seed_indices = ordered[:SEED_LIMIT]
    seed_indices_by_intent[slug] = seed_indices.tolist()
    centroid = normalized_vectors[seed_indices].mean(axis=0)
    centroid = centroid / np.clip(np.linalg.norm(centroid), 1e-12, None)
    centroids.append(centroid.astype(np.float32))

centroids = np.vstack(centroids)
print('시드 수:', {slug: len(indices) for slug, indices in seed_indices_by_intent.items()})

## 4. 전체 기사 분류

의미 유사도 75%와 규칙 근거 25%를 결합합니다. 2위 점수가 1위와 가깝다면 보조 의도를 함께 기록합니다.

In [ ]:
semantic_scores = np.clip(normalized_vectors @ centroids.T, 0.0, 1.0)
rule_scales = np.maximum(raw_rule_scores.max(axis=0, keepdims=True), 1.0)
normalized_rule_scores = np.clip(raw_rule_scores / rule_scales, 0.0, 1.0)
intent_scores = 0.75 * semantic_scores + 0.25 * normalized_rule_scores

labels = []
for row_index, article in enumerate(articles):
    order = np.argsort(intent_scores[row_index])[::-1]
    primary_index, secondary_index = int(order[0]), int(order[1])
    primary_slug = INTENT_SLUGS[primary_index]
    primary_score = float(intent_scores[row_index, primary_index])
    secondary_score = float(intent_scores[row_index, secondary_index])
    margin = primary_score - secondary_score
    secondary_slug = ''
    if secondary_score >= SECONDARY_MIN_SCORE and margin <= SECONDARY_MARGIN:
        secondary_slug = INTENT_SLUGS[secondary_index]
    confidence = float(np.clip(0.7 * primary_score + 0.3 * min(1.0, margin / 0.15), 0.0, 1.0))
    labels.append({
        **article,
        'primary_intent': primary_slug,
        'secondary_intents': secondary_slug,
        'confidence': confidence,
        'label_method': 'rule_seed_centroid_v1',
        'intent_scores': {
            slug: float(intent_scores[row_index, idx])
            for idx, slug in enumerate(INTENT_SLUGS)
        },
    })

print('주 의도 분포:', Counter(item['primary_intent'] for item in labels))

## 5. 의도별 대표 기사 선택

의도 적합도와 벡터 다양성을 함께 고려하며, 의미 중복도가 0.94 이상인 기사는 같은 참조 묶음에서 반복 선택하지 않습니다.

In [ ]:
def select_references(intent_index: int, slug: str) -> list[int]:
    candidate_indices = [
        index
        for index, item in enumerate(labels)
        if item['primary_intent'] == slug or item['secondary_intents'] == slug
    ]
    candidate_indices.sort(key=lambda index: intent_scores[index, intent_index], reverse=True)
    if not candidate_indices:
        return []

    selected = []
    remaining = candidate_indices.copy()
    while remaining and len(selected) < REFERENCE_LIMIT:
        if not selected:
            best = remaining.pop(0)
            selected.append(best)
            continue

        remaining_array = np.asarray(remaining, dtype=int)
        similarities = normalized_vectors[remaining_array] @ normalized_vectors[selected].T
        max_similarity = similarities.max(axis=1)
        relevance = intent_scores[remaining_array, intent_index]
        mmr = MMR_RELEVANCE * relevance - (1.0 - MMR_RELEVANCE) * max_similarity
        valid = np.flatnonzero(max_similarity < DUPLICATE_COSINE)
        if len(valid) == 0:
            break
        best_position = int(valid[np.argmax(mmr[valid])])
        best = remaining.pop(best_position)
        selected.append(best)
    return selected

reference_indices = {}
for intent_index, slug in enumerate(INTENT_SLUGS):
    reference_indices[slug] = select_references(intent_index, slug)

reference_memberships = {index: [] for index in range(len(labels))}
for slug, indices in reference_indices.items():
    for index in indices:
        reference_memberships[index].append(slug)

print('참조 기사 수:', {slug: len(indices) for slug, indices in reference_indices.items()})

## 6. 검토용 CSV와 추천용 벡터 저장

In [ ]:
score_columns = [f'score_{slug}' for slug in INTENT_SLUGS]
fieldnames = [
    'source_vector_index', 'article_id', 'digest_date', 'title', 'summary',
    'publisher', 'article_link', 'primary_intent', 'primary_intent_label',
    'secondary_intents', 'secondary_intent_labels', 'confidence',
    'label_method', 'use_as_reference', 'reference_intents',
    *score_columns,
]

with LABEL_CSV_PATH.open('w', encoding='utf-8-sig', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    for index, item in enumerate(labels):
        secondary = item['secondary_intents']
        row = {
            'source_vector_index': index,
            'article_id': item.get('article_id', ''),
            'digest_date': item.get('digest_date', ''),
            'title': item.get('title', ''),
            'summary': item.get('summary', ''),
            'publisher': item.get('publisher', ''),
            'article_link': item.get('article_link', ''),
            'primary_intent': item['primary_intent'],
            'primary_intent_label': INTENTS[item['primary_intent']]['label'],
            'secondary_intents': secondary,
            'secondary_intent_labels': INTENTS[secondary]['label'] if secondary else '',
            'confidence': f"{item['confidence']:.6f}",
            'label_method': item['label_method'],
            'use_as_reference': bool(reference_memberships[index]),
            'reference_intents': ';'.join(reference_memberships[index]),
        }
        for slug in INTENT_SLUGS:
            row[f'score_{slug}'] = f"{item['intent_scores'][slug]:.6f}"
        writer.writerow(row)

intent_metadata = {'classification_version': CLASSIFICATION_VERSION, 'intents': {}}
for slug, indices in reference_indices.items():
    vector_path = INTENT_DIR / f'{slug}_embeddings.npy'
    intent_vectors = np.asarray(normalized_vectors[indices], dtype=np.float32)
    np.save(vector_path, intent_vectors)
    intent_metadata['intents'][slug] = {
        'label': INTENTS[slug]['label'],
        'description': INTENTS[slug]['description'],
        'embedding_file': vector_path.name,
        'rows': [
            {
                'embedding_row': embedding_row,
                'source_vector_index': source_index,
                'article_id': labels[source_index].get('article_id', ''),
                'title': labels[source_index].get('title', ''),
                'article_link': labels[source_index].get('article_link', ''),
                'intent_score': labels[source_index]['intent_scores'][slug],
            }
            for embedding_row, source_index in enumerate(indices)
        ],
    }

METADATA_PATH.write_text(
    json.dumps(intent_metadata, ensure_ascii=False, indent=2), encoding='utf-8'
)

manifest = {
    'classification_version': CLASSIFICATION_VERSION,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'source_embedding_file': SOURCE_VECTOR_PATH.name,
    'source_metadata_file': SOURCE_METADATA_PATH.name,
    'source_rows': len(labels),
    'dimension': int(normalized_vectors.shape[1]),
    'model': metadata_document.get('model', ''),
    'source_weights': metadata_document.get('weights', {}),
    'method': 'high_precision_rule_seeds + centroid cosine + MMR references',
    'semantic_weight': 0.75,
    'rule_weight': 0.25,
    'reference_limit': REFERENCE_LIMIT,
    'duplicate_cosine': DUPLICATE_COSINE,
    'label_csv': str(LABEL_CSV_PATH.relative_to(MODEL_DIR)),
    'metadata_file': METADATA_PATH.name,
    'intents': {
        slug: {
            'label': INTENTS[slug]['label'],
            'reference_rows': len(reference_indices[slug]),
            'primary_rows': sum(item['primary_intent'] == slug for item in labels),
        }
        for slug in INTENT_SLUGS
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

print('라벨 CSV:', LABEL_CSV_PATH)
print('의도 임베딩:', INTENT_DIR)
print(json.dumps(manifest['intents'], ensure_ascii=False, indent=2))

## 7. 검토 지침

생성된 CSV에서 다음 항목을 우선 검토합니다.

1. `confidence`가 낮은 기사
2. 주·보조 의도가 함께 지정된 기사
3. `use_as_reference=true`인 기사
4. 특정 기업·사건이 참조 묶음에 반복되는지

CSV를 사람이 수정한 뒤 다시 벡터를 뽑는 단계는 이 노트북의 후속 버전으로 분리합니다.